In [1]:
!pip install chromadb langchain langchain-community rank_bm25 requests

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached rank_bm25-0.2.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_classic-1.0.4-py3-none-any.whl.metadata (4.8 kB)
  Using cached sqlalchemy-2.0.49-cp314-cp314-macosx_11_0_arm64.whl.metadata (9.5 kB)
  Using cached aiohttp-3.13.5-cp314-cp314-macosx_11_0_arm64.whl.metadata (8.1 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached marshmallow-3.26.2-py3-none-any.whl.metadata (7.3 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
  Using cached mypy_extensions-1.1.0-py3-none-any.whl.metadata (1.1 kB)
Using cached langchain_community-0.4.1-py3-none

In [5]:
#################################
# Hybrid Retriever (Chroma + Ollama + BM25)
#################################

print("="*40)
# Imports
print("="*40)

import requests
print("----------------- requests completed or connected, ---------")

import chromadb
print("----------------- chromadb completed or connected, ---------")

from rank_bm25 import BM25Okapi
print("----------------- rank_bm25 completed or connected, ---------")

from typing import List
print("----------------- typing completed or connected, ---------")

import time
print("----------------- time completed or connected, ---------")


print("="*40)
# Step 1: Connect to Chroma
print("="*40)

print("#===============[ Step 1: Connect to Chroma ]==========")

client = chromadb.HttpClient(host="192.168.1.30", port=8000)
collection = client.get_collection(name="dev")


print("="*40)
# Step 2: Ollama Embedding
print("="*40)

print("#===============[ Step 2: Ollama Embedding ]==========")

OLLAMA_URL = "http://localhost:11434/api/embeddings"
MODEL_NAME = "embeddinggemma:300m"


def get_embedding(text: str) -> List[float]:
    """Get embedding from Ollama."""
    response = requests.post(
        OLLAMA_URL,
        json={"model": MODEL_NAME, "prompt": text}
    )
    return response.json()["embedding"]


print("="*40)
# Step 3: Load Documents for BM25
print("="*40)

print("#===============[ Step 3: Load Documents for BM25 ]==========")

data = collection.get()

documents = data["documents"]
ids = data["ids"]

# Tokenize for BM25
tokenized_docs = [doc.split() for doc in documents]

bm25 = BM25Okapi(tokenized_docs)


print("="*40)
# Step 4: Hybrid Search Function
print("="*40)

print("#===============[ Step 4: Hybrid Search Function ]==========")


def hybrid_search(query: str, top_k: int = 5, alpha: float = 0.7):
    """Hybrid search using dense + BM25."""

    print("#===============[ Hybrid Retrieval Started ]==========")
    start_time = time.perf_counter()

    # -------- Dense Search --------
    t0 = time.perf_counter()

    query_embedding = get_embedding(query)

    t1 = time.perf_counter()

    dense_results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    t2 = time.perf_counter()

    dense_docs = dense_results["documents"][0]
    dense_distances = dense_results["distances"][0]

    # Convert to similarity
    dense_map = {
        doc: (1 - dist) for doc, dist in zip(dense_docs, dense_distances)
    }

    # -------- BM25 --------
    tokenized_query = query.split()
    bm25_scores = bm25.get_scores(tokenized_query)

    # Fix NumPy issue
    max_bm25 = bm25_scores.max() if len(bm25_scores) > 0 else 1
    bm25_scores = [s / max_bm25 for s in bm25_scores]

    t3 = time.perf_counter()

    # -------- Hybrid Combine --------
    hybrid_scores = []
    for i, doc in enumerate(documents):
        dense_score = dense_map.get(doc, 0)
        score = alpha * dense_score + (1 - alpha) * bm25_scores[i]
        hybrid_scores.append((doc, score))

    hybrid_scores.sort(key=lambda x: x[1], reverse=True)

    end_time = time.perf_counter()

    # -------- Timing Logs --------
    print(f"Embedding time: {t1 - t0:.4f}s")
    print(f"Dense search time: {t2 - t1:.4f}s")
    print(f"BM25 time: {t3 - t2:.4f}s")
    print(f"#===============[ Total Retrieval Time: {end_time - start_time:.4f} sec ]==========")

    return hybrid_scores[:top_k]


print("="*40)
# Step 5: Test Query
print("="*40)

print("#===============[ Step 5: Test Query ]==========")

results = hybrid_search("laser sensor is not working what i need to check?", top_k=5)

for i, (doc, score) in enumerate(results):
    print(f"\nResult {i+1} | Score: {score:.4f}")
    print(doc[:200])


print("#===============[ process completed]==========")

----------------- requests completed or connected, ---------
----------------- chromadb completed or connected, ---------
----------------- rank_bm25 completed or connected, ---------
----------------- typing completed or connected, ---------
----------------- time completed or connected, ---------
#===============[ Step 1: Connect to Chroma ]==========
#===============[ Step 2: Ollama Embedding ]==========
#===============[ Step 3: Load Documents for BM25 ]==========
#===============[ Step 4: Hybrid Search Function ]==========
#===============[ Step 5: Test Query ]==========
#===============[ Hybrid Retrieval Started ]==========
Embedding time: 0.1389s
Dense search time: 0.3623s
BM25 time: 0.0105s
#===============[ Total Retrieval Time: 0.5159 sec ]==========

Result 1 | Score: 0.3505
note : height of dispense primer need to check with 3d laser at the below regions.

Result 2 | Score: 0.3360
bhdp path height check through 3d laser note : height of dispense primer need to check with 3d

In [6]:
print("="*40)
# Step 5: Multi Query Benchmark
print("="*40)

print("#===============[ Step 5: Benchmark 5 Queries ]==========")

queries = [
    "laser sensor is not working what i need to check?",
    "what is machine learning?",
    "how to fix motor overload issue?",
    "what are safety precautions in CNC machine?",
    "why is temperature sensor failing?"
]

times = []

for q in queries:
    print("\n----------------------------------------")
    print(f"Query: {q}")

    start = time.perf_counter()

    results = hybrid_search(q, top_k=5)

    end = time.perf_counter()
    elapsed = end - start
    times.append(elapsed)

    print(f"⏱️ Query Time: {elapsed:.4f} sec")

    for i, (doc, score) in enumerate(results):
        print(f"\nResult {i+1} | Score: {score:.4f}")
        print(doc[:150])


# -------- Average Time --------
avg_time = sum(times) / len(times)

print("\n========================================")
print(f"Average Retrieval Time (5 queries): {avg_time:.4f} sec")
print("========================================")

print("#===============[ process completed]==========")

#===============[ Step 5: Benchmark 5 Queries ]==========

----------------------------------------
Query: laser sensor is not working what i need to check?
#===============[ Hybrid Retrieval Started ]==========
Embedding time: 0.1648s
Dense search time: 4.1105s
BM25 time: 0.0182s
#===============[ Total Retrieval Time: 4.2972 sec ]==========
⏱️ Query Time: 4.2977 sec

Result 1 | Score: 0.3505
note : height of dispense primer need to check with 3d laser at the below regions.

Result 2 | Score: 0.3360
bhdp path height check through 3d laser note : height of dispense primer need to check with 3d laser at the below regions.

Result 3 | Score: 0.2953
summary : this document details " user login activity ". the page is not a technical drawing in the traditional sense but rather a description of the 

Result 4 | Score: 0.2917
working principle 1. the hiwin driver controls the brake through its i / o cable. 2. when the drive is enabled, the driver sends a brake release signa

Result 5 | Score